# Analisis 500 kbit/s con distintos tamanos de paquete

En estas pruebas se mide el RTT y la perdida usando `ping` hacia M2 y M3. Se comparan paquetes de 120, 512 y 1024 bytes en dos condiciones: sin limitacion y con ancho de banda limitado a 500 kbit/s mediante `tc`.

In [ ]:
from pathlib import Path
import csv
import re

BASE_DIR = Path.cwd()
if not list(BASE_DIR.glob('*.txt')):
    BASE_DIR = Path('host/metricas_network_path/medidas_tc/500kbits_tamaños_diferentes')

OUT_DIR = BASE_DIR / 'graficas_y_resumen'
OUT_DIR.mkdir(exist_ok=True)

def parse_file(path):
    text = path.read_text(encoding='utf-8', errors='ignore')
    size_match = re.search(r'_(\d+)bytes', path.name)
    size = int(size_match.group(1)) if size_match else None
    condition = '500 kbit/s' if '500kbit' in path.name else 'Sin limitacion'

    rows = []
    pattern = re.compile(
        r'Ping .*?(M[23]).*?--- .*? ping statistics ---\n'
        r'(\d+) packets transmitted, (\d+) received, ([\d.]+)% packet loss.*?'
        r'rtt min/avg/max/mdev = ([\d.]+)/([\d.]+)/([\d.]+)/([\d.]+) ms',
        re.S,
    )
    for match in pattern.finditer(text):
        dest, tx, rx, loss, rtt_min, rtt_avg, rtt_max, mdev = match.groups()
        rows.append({
            'archivo': path.name,
            'condicion': condition,
            'tamano_bytes': size,
            'destino': dest,
            'transmitidos': int(tx),
            'recibidos': int(rx),
            'perdida_pct': float(loss),
            'rtt_min_ms': float(rtt_min),
            'rtt_avg_ms': float(rtt_avg),
            'rtt_max_ms': float(rtt_max),
            'mdev_ms': float(mdev),
        })
    return rows

rows = []
for path in sorted(BASE_DIR.glob('*.txt')):
    rows.extend(parse_file(path))

summary_path = OUT_DIR / 'resumen_500kbits.csv'
with summary_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

rows

In [ ]:
def save_svg_bar(path, title, y_label, data, y_key, width=1040, height=560):
    margin = 78
    plot_w = width - margin * 2
    plot_h = height - margin * 2
    max_y = max(item[y_key] for item in data) or 1
    max_y *= 1.2
    bar_w = plot_w / len(data) * 0.68

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width/2}" y="32" text-anchor="middle" font-family="Arial" font-size="22">{title}</text>',
        f'<text x="22" y="{height/2}" transform="rotate(-90 22 {height/2})" text-anchor="middle" font-family="Arial" font-size="14">{y_label}</text>',
        f'<line x1="{margin}" y1="{height-margin}" x2="{width-margin}" y2="{height-margin}" stroke="#333"/>',
        f'<line x1="{margin}" y1="{margin}" x2="{margin}" y2="{height-margin}" stroke="#333"/>',
    ]

    for i in range(6):
        y_val = max_y * i / 5
        y = height - margin - (y_val / max_y) * plot_h
        parts.append(f'<line x1="{margin}" y1="{y:.1f}" x2="{width-margin}" y2="{y:.1f}" stroke="#ddd"/>')
        parts.append(f'<text x="{margin-8}" y="{y+4:.1f}" text-anchor="end" font-family="Arial" font-size="12">{y_val:.2f}</text>')

    colors = {'M2': '#1971c2', 'M3': '#f08c00'}
    for idx, item in enumerate(data):
        x_center = margin + (idx + 0.5) * plot_w / len(data)
        bar_h = (item[y_key] / max_y) * plot_h
        x = x_center - bar_w / 2
        y = height - margin - bar_h
        label = f"{item['condicion']}\n{item['tamano_bytes']}B {item['destino']}"
        fill = colors.get(item['destino'], '#555')
        parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="{fill}"/>')
        parts.append(f'<text x="{x_center:.1f}" y="{y-6:.1f}" text-anchor="middle" font-family="Arial" font-size="11">{item[y_key]:.3f}</text>')
        for line_no, line in enumerate(label.split('\n')):
            parts.append(f'<text x="{x_center:.1f}" y="{height-margin+18+line_no*14}" text-anchor="middle" font-family="Arial" font-size="10">{line}</text>')

    parts.append('</svg>')
    path.write_text('\n'.join(parts), encoding='utf-8')

ordered = sorted(rows, key=lambda r: (r['tamano_bytes'], r['condicion'] != 'Sin limitacion', r['destino']))
save_svg_bar(OUT_DIR / 'grafica_rtt_medio_500kbits.svg', 'RTT medio por condicion y tamano', 'RTT medio (ms)', ordered, 'rtt_avg_ms')
save_svg_bar(OUT_DIR / 'grafica_rtt_maximo_500kbits.svg', 'RTT maximo por condicion y tamano', 'RTT maximo (ms)', ordered, 'rtt_max_ms')
save_svg_bar(OUT_DIR / 'grafica_perdida_500kbits.svg', 'Perdida por condicion y tamano', 'Perdida (%)', ordered, 'perdida_pct')

[str(OUT_DIR / name) for name in ['grafica_rtt_medio_500kbits.svg', 'grafica_rtt_maximo_500kbits.svg', 'grafica_perdida_500kbits.svg']]